# MCGS → geometric GS → conformal 2D mesh

Tracked as `confMesh2d_mcgs.ipynb`.

Teaching path from a Monte-Carlo labelled image to a grain-boundary-conformant FE mesh.

Uses **`confMesh2dGMSH`** (raw Gmsh). The pygmsh class `confMesh2d` is deprecated.

For a mesh-only walkthrough (no MCGS), see `confMesh2d_gmsh.ipynb`. Requires `gmsh` (`pip install upxo[mesh]`).

## STEP 1 — Grain structure generation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from upxo.ggrowth.mcgs import mcgs
from upxo.pxtal.geometrification import polygonised_grain_structure
from upxo.meshing.conformal_mesher2d import confMesh2dGMSH
from upxo.meshing.writer_ABQ import summarize_inp
from pathlib import Path
_XLS = next(
    (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls"
     for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src" / "upxo" / "demos" / "confMesh" / "confMesh1.xls").is_file()),
    Path("confMesh1.xls"),
)
print("dashboard:", _XLS)

In [ ]:
pxt = mcgs(input_dashboard=str(_XLS))
pxt.simulate()
pxt.detect_grains(library="cc3d", connectivity=4)
gstslice = pxt.gs[list(pxt.gs.keys())[-1]]
gstslice.char_morph_2d(
    bbox=True, use_version=2, bbox_ex=True, area=True, eq_diameter=True,
    perimeter=True, aspect_ratio=True, solidity=True, feret_diameter=True,
    saa=True, char_gb=True, make_skim_prop=True, throw=False, append=False)
gstslice.find_neigh()
print("grains", len(np.unique(gstslice.lgi)) - 1)

## STEP 2 — Pixelated GS to geometric GS

`polygonised_grain_structure` parses `lgi` / grain IDs / neighbours into per-grain polygons and a GB network.

In [ ]:
geomGS = polygonised_grain_structure(gstslice.lgi, gstslice.gid, gstslice.neigh_gid)
geomGS.pix_to_geom(verbose=False, perform_xtal_dict_check=True)
fig, ax = geomGS.plotgs(geomGS.POLYXTAL, figsize=(5, 5), dpi=110)
ax.set_title("raw polygons")
fig

## STEP 3 — Grain-boundary smoothing

Smoothing shortens stair-step pixel boundaries so Gmsh can place conformal GB edges.

In [ ]:
npasses = 2
min_segment_length_factor = 2
gsname = f"gs.{npasses}passes.{min_segment_length_factor}minseglenfactor"
geomGS.smooth_gbsegs(
    geomGS.GB, npasses=npasses,
    max_smooth_levels=np.repeat(min_segment_length_factor, npasses),
    plot=False, name=gsname)
pxtal = geomGS.smoothed[gsname]["POLYXTAL"]
fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=120)
geomGS.plotgs(geomGS.POLYXTAL, fig=fig, ax=axes[0])
axes[0].set_title("raw")
axes[0].set_axis_off()
geomGS.plotgs(pxtal, fig=fig, ax=axes[1])
axes[1].set_title("smoothed")
axes[1].set_axis_off()
fig

## STEP 4 — Conformal FE mesh (`confMesh2dGMSH`)

`from_geometric_pxtal` is the drop-in replacement for the old pygmsh constructor. `femesh_gmsh` talks to the raw Gmsh API.

In [ ]:
gsConfMesh = confMesh2dGMSH.from_geometric_pxtal(
    gsgen_method="shapely_pxtal_load",
    pxtal=pxtal,
    xbound=[gstslice.uigrid.xmin, gstslice.uigrid.xmax],
    ybound=[gstslice.uigrid.ymin, gstslice.uigrid.ymax])
gsConfMesh.femesh_gmsh(
    mesh_size_gb=1.0, mesh_size_bulk=2.0,
    mesh_algo=8, mesh_order=1, recombine_to_quads=True)
gsConfMesh.form_elsets_gmsh()
gsConfMesh.build_boundary_nsets()
gsConfMesh.build_gb_nset()
print(gsConfMesh.validation_report)

Grain-coloured fill and face NSETs come from `plot_by_grain` (`upxo.viz.meshviz`). Abaqus export is `export_abaqus_inp` (`upxo.meshing.writer_ABQ`): CPS4/CPS3 for `plane='stress'`, CPE4/CPE3 for `'strain'`.

In [ ]:
fig, ax = gsConfMesh.plot_by_grain(
    figsize=(6, 6), show_gb=True, show_nsets=True,
    title="MCGS conformal mesh by grain")
fig

In [ ]:
out = Path.cwd() / "confMesh2d_mcgs_out"
inp = gsConfMesh.export_abaqus_inp(out / "rve_cps.inp", plane="stress")
inp, summarize_inp(inp)